# UDF in pyspark
* UDFs allow you to define **custom transformations** in Spark using Python, Scala, or Java.
* They operate on data **record by record**.
* By default, they are registered as **temporary functions** within a specific `SparkSession` or context.
* UDFs can take one or more columns as input and return one or more outputs.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.master("local[*]").appName("udfSpark").getOrCreate()

### creating DataFrame

In [2]:
data = [(1, 'maheer', 2000, 5000), (1, 'ravi', 4000, 1000)]
schema = ['id', 'name', 'salary', 'bonus']
df = spark.createDataFrame(data, schema)
df.show()

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|maheer|  2000| 5000|
|  1|  ravi|  4000| 1000|
+---+------+------+-----+



## 1. Registering UDF as DataFrame Functions

In [3]:
def totalPay(sal, bon):
    return sal + bon

# creating udf using below 
totalPayment = udf(lambda s, b: totalPay(s, b), IntegerType())

In [4]:
help(udf)

Help on function udf in module pyspark.sql.functions:

udf(f: Union[Callable[..., Any], ForwardRef('DataTypeOrString'), NoneType] = None, returnType: 'DataTypeOrString' = StringType()) -> Union[ForwardRef('UserDefinedFunctionLike'), Callable[[Callable[..., Any]], ForwardRef('UserDefinedFunctionLike')]]
    Creates a user defined function (UDF).
    
    .. versionadded:: 1.3.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    f : function
        python function if used as a standalone function
    returnType : :class:`pyspark.sql.types.DataType` or str
        the return type of the user-defined function. The value can be either a
        :class:`pyspark.sql.types.DataType` object or a DDL-formatted type string.
    
    Examples
    --------
    >>> from pyspark.sql.types import IntegerType
    >>> slen = udf(lambda s: len(s), IntegerType())
    >>> @udf
    ... def to_upper(s):
    ...     if s is not None:
    ...         retu

In [5]:
# using udf
df.withColumn("totalPay", totalPay((df.salary), (df.bonus))).show()

+---+------+------+-----+--------+
| id|  name|salary|bonus|totalPay|
+---+------+------+-----+--------+
|  1|maheer|  2000| 5000|    7000|
|  1|  ravi|  4000| 1000|    5000|
+---+------+------+-----+--------+



### 1.1 Registering UDF as DataFrame Functions uding annotation

In [6]:
# register the udf using annotations
@udf(returnType=IntegerType())
def total(sal, bon):
    return sal + bon

In [7]:
df.select('*', totalPay(col("salary"), df.bonus).alias('totalPay')).show()

+---+------+------+-----+--------+
| id|  name|salary|bonus|totalPay|
+---+------+------+-----+--------+
|  1|maheer|  2000| 5000|    7000|
|  1|  ravi|  4000| 1000|    5000|
+---+------+------+-----+--------+



## 2. Registering UDF as SQL function

In [8]:
# create temp view in the session 
df.createOrReplaceTempView('emp')
spark.sql("select * from emp").show()

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|maheer|  2000| 5000|
|  1|  ravi|  4000| 1000|
+---+------+------+-----+



In [9]:
# Registering UDF as SQL function
def total(sal, bon):
    return sal + bon

spark.udf.register(name='TotalPaySQL', f=totalPay, returnType=IntegerType())

<function __main__.totalPay(sal, bon)>

In [10]:
spark.sql("Select name, TotalPaySQL(salary, bonus) as totPay from emp").show()

+------+------+
|  name|totPay|
+------+------+
|maheer|  7000|
|  ravi|  5000|
+------+------+



## Performance-Considerations

* **Scala/Java UDFs**: Run inside the JVM → faster, but cannot use Spark’s built-in code generation optimizations.
* **Python UDFs**: Data must be serialized between JVM and Python → slower, higher memory cost, potential worker failures.

  * Recommendation: Prefer **Scala/Java UDFs** for performance.

## Key-Takeaways

1. UDFs let you extend Spark with **custom logic**.
2. **Scala/Java UDFs** → better performance.
3. Always **specify return type**.
4. UDFs can be used in **DataFrame API or SQL**.
5. Hive support allows **Hive-style UDFs**.